In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# 🚀 Section 2: The Naive Inference Loop — An O(n³) Disaster

*Part of the Vizuara series on KV Caching and Efficient Inference*
*Estimated time: 20–30 minutes*

## 1. Why Does This Matter?

Imagine you are building a product powered by a large language model — a legal assistant, a code copilot, a customer support bot. Your model is brilliant. But users are complaining that it feels *slow*. Every time it generates a word, it seems to pause and think. You profile the system and discover something startling: the model is re-reading the entire conversation from scratch every single time it generates a new token.

That is not a bug in your code. That is the **naive inference loop**, and it is the villain of this notebook.

By the end of this section, you will:

- Understand *exactly* what the attention mechanism does at each generation step
- Measure the computational cost of naive inference and watch it grow with sequence length
- See with your own eyes why this approach scales as **O(n³)** — a disaster for long sequences
- Build a clear intuition for *why* this happens before we introduce the fix (KV caching) in the next section

Here is a teaser of what we will produce — a timing curve that makes the problem undeniable:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Teaser: What does naive inference cost look like?
seq_lengths = np.array([128, 256, 512, 1024, 2048, 4096])
# Hypothetical cubic scaling: cost ~ n^3
cubic_cost = (seq_lengths / 128) ** 3

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(seq_lengths, cubic_cost, 'o-', color='crimson', linewidth=2.5,
        markersize=8, label='Naive inference cost ∝ n³')
ax.fill_between(seq_lengths, cubic_cost, alpha=0.15, color='crimson')
ax.set_xlabel('Sequence Length (tokens)', fontsize=13)
ax.set_ylabel('Relative Compute Cost', fontsize=13)
ax.set_title('Teaser: Naive Inference Scales as O(n³)\n(We will derive this from first principles)', fontsize=13)
ax.legend(fontsize=12)
ax.set_yscale('log')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('teaser_cubic_scaling.png', dpi=120)
plt.show()
print("This is where we are headed. Let us understand WHY.")

## 2. Building Intuition

Before we touch a single equation, let us build a crystal-clear mental model of what is actually happening inside a transformer when it generates a token.

### The Library Analogy

Picture a scholar sitting in an enormous library. The library contains every book written so far in a conversation — let us say there are **n books** on the table. The scholar's job is to write the next sentence.

Here is what the scholar does in the **naive** approach:

1. **For every book already on the table**, the scholar reads it again from cover to cover and asks: *"How relevant is this book to what I am about to write?"* This produces a **relevance score** for each book.
2. The scholar then **blends** all the books together, weighted by those relevance scores, into a single synthesis.
3. From that synthesis, the scholar writes the **next word**.
4. That word becomes a new book and is placed on the table.
5. For the *next* word, the scholar goes back to step 1 — and now there are **n+1 books** to read.

The key insight: **the scholar never remembers what they learned from reading**. Every single word they write requires re-reading every book from scratch. This is wasteful in exactly the way that makes your stomach drop.

### What "Attention" Actually Does, in Plain English

Attention is the mechanism by which a transformer token asks: *"Of all the other tokens I can see, which ones should I pay attention to, and how much?"*

Concretely, for each token in the sequence, attention does three things:

| Role | What it represents | Analogy |
|------|-------------------|---------|
| **Query (Q)** | "What am I looking for?" | The scholar's research question |
| **Key (K)** | "What do I offer?" | The title/index of each book |
| **Value (V)** | "What is my actual content?" | The full text of each book |

To generate token number **n+1**, the transformer takes the query of the new token and compares it to the **keys of all n previous tokens**. This comparison (a dot product) produces **n scores**. Those scores are used to take a weighted sum of all **n values**. That weighted sum is the information the new token "sees."

The critical word is **all**. Every generation step touches every prior token, every single time.

### The Scaling Trap

Now think about what happens as the sequence grows:

- At step 100, generating one token requires looking at 100 prior tokens → **100 operations**
- At step 1,000, generating one token requires looking at 1,000 prior tokens → **1,000 operations**
- At step 10,000 → **10,000 operations**

But this only counts the cost of *one* generation step. To generate an entire sequence of length **n**, you must pay this cost **n times** — once for each new token. So the total work is roughly:

$$1 + 2 + 3 + \ldots + n = \frac{n(n+1)}{2} \approx O(n^2)$$

That is already quadratic. But attention is not just a single scalar comparison — it operates across **d** dimensions (the embedding dimension), and the matrix multiplications involved scale that cost by **d** as well. For a sequence of length n with embedding dimension d, the full attention computation is:

$$O(n^2 \cdot d)$$

In practice, for fixed model size d is a constant, so we call this **O(n²)** in sequence length. When you also consider that for each step we recompute Q, K, V projections (which are themselves O(n·d²)), the total cost of naively generating n tokens becomes **O(n³)** — and that is the disaster.

### 🤔 Think About This

> Before you run any code, pause and reason through this:
>
> Suppose generating a 100-token response takes 1 second with the naive approach.
> Roughly how long would you expect a 1,000-token response to take?
> What about 10,000 tokens?
>
> Write your guesses here before you see the empirical measurements.
> (Hint: if naive scaling were linear, 1,000 tokens would take 10 seconds. But it is *not* linear...)

## 3. The Mathematics

Let us be precise. We will build up the cost calculation from first principles, step by step.

### 3.1 The Attention Equation

For a single attention head, the output is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

**What this equation says:** Take the query matrix Q (shape: n × d_k), multiply it by the transpose of the key matrix K (shape: d_k × n), divide by √d_k to stabilize gradients, apply softmax to get probabilities, then use those probabilities to take a weighted combination of the values V (shape: n × d_v).

**Computationally, this means:**
- QKᵀ is a matrix multiply of shapes (n × d_k) · (d_k × n) → result is (n × n). Cost: **O(n² · d_k)**
- The softmax is applied to an n×n matrix. Cost: **O(n²)**
- Multiplying the n×n attention weights by V (n × d_v) → result (n × d_v). Cost: **O(n² · d_v)**

**Total cost of one attention operation:** O(n² · d), where d = d_k = d_v.

### 3.2 The Cost of Naive Autoregressive Generation

Now here is where the disaster unfolds. In autoregressive generation, we generate one token at a time. At step **t**, we have generated t-1 tokens and want to generate token t.

**The naive approach recomputes everything from scratch.** So at step t, the cost is:

$$C(t) = O(t^2 \cdot d)$$

because we run full attention over a sequence of length t.

**Total cost to generate n tokens:**

$$C_{\text{total}} = \sum_{t=1}^{n} C(t) = \sum_{t=1}^{n} O(t^2 \cdot d) = O\!\left(d \sum_{t=1}^{n} t^2\right) = O\!\left(d \cdot \frac{n(n+1)(2n+1)}{6}\right) = O(n^3 \cdot d)$$

**This equation says:** The total work grows as the *cube* of sequence length. Double the sequence length → 8× the compute. Triple it → 27× the compute. This is the O(n³) disaster.

### 3.3 The QKV Projection Cost

There is an additional cost we must account for. To get Q, K, V in the first place, each token's embedding (dimension d_model) is projected through three weight matrices W_Q, W_K, W_V (each of shape d_model × d_k).

For a sequence of length n, these projections cost:

$$C_{\text{proj}} = O(n \cdot d_{\text{model}} \cdot d_k)$$

Since d_model and d_k are fixed model hyperparameters (constants), this is **O(n)** in sequence length — linear, and therefore not the dominant term. The attention matrix computation O(n²·d) dominates for long sequences.

**Key takeaway:** The n×n attention matrix is the heart of the scaling problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let us verify the math by computing these costs symbolically
d = 64       # head dimension (a typical value)
d_model = 512

seq_lengths = np.array([64, 128, 256, 512, 1024, 2048, 4096])

# Cost per step at step t: O(t^2 * d)
# Total naive cost: sum_{t=1}^{n} t^2 * d  (the cubic term)
naive_total = np.array([d * sum(t**2 for t in range(1, n+1)) for n in seq_lengths])

# Compare to linear and quadratic for reference
linear_cost  = d * seq_lengths
quadratic_cost = d * seq_lengths**2
cubic_approx = d * seq_lengths**3 / 3  # the analytic approximation

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Linear scale ---
ax = axes[0]
ax.plot(seq_lengths, naive_total / naive_total[0],
        'o-', color='crimson', lw=2.5, ms=8, label='Naive total (exact sum)')
ax.plot(seq_lengths, (seq_lengths / seq_lengths[0])**3,
        '--', color='orange', lw=2, label='O(n³) reference')
ax.plot(seq_lengths, (seq_lengths / seq_lengths[0])**2,
        '--', color='steelblue', lw=2, label='O(n²) reference')
ax.plot(seq_lengths, (seq_lengths / seq_lengths[0]),
        '--', color='green', lw=2, label='O(n) reference')
ax.set_xlabel('Sequence Length n', fontsize=12)
ax.set_ylabel('Relative Cost (normalized to n=64)', fontsize=12)
ax.set_title('Naive Inference: Total Compute Cost', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Right: Log-log scale to see slopes ---
ax = axes[1]
ax.loglog(seq_lengths, naive_total / naive_total[0],
          'o-', color='crimson', lw=2.5, ms=8, label='Naive total (exact sum)')
ax.loglog(seq_lengths, (seq_lengths / seq_lengths[0])**3,
          '--', color='orange', lw=2, label='Slope = 3  (O(n³))')
ax.loglog(seq_lengths, (seq_lengths / seq_lengths[0])**2,
          '--', color='steelblue', lw=2, label='Slope = 2  (O(n²))')
ax.set_xlabel('Sequence Length n (log scale)', fontsize=12)
ax.set_ylabel('Relative Cost (log scale)', fontsize=12)
ax.set_title('Log-Log Plot: Confirming the Cubic Slope', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Mathematical Verification of O(n³) Naive Inference Cost',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('math_verification.png', dpi=120)
plt.show()

print("On the log-log plot, a slope of 3 confirms O(n³) scaling.")
print(f"At n=4096, naive cost is {naive_total[-1]/naive_total[0]:.1f}× more than at n=64.")
print(f"That is {naive_total[-1]/naive_total[0] / (4096/64)**2:.1f}× worse than quadratic!")

## 4. Let's Build It — Component by Component

Now we will implement a naive transformer inference loop from scratch so we can *measure* the disaster empirically. We will build three components:

1. **Single-head attention** — the core operation
2. **A naive token generator** — re-runs attention from scratch at every step
3. **A timing harness** — measures wall-clock time vs. sequence length

### 4.1 Single-Head Scaled Dot-Product Attention

Let us implement the raw attention mechanism. We will do this without any masking optimizations — just the bare computation.

In [ ]:
import torch
import torch.nn.functional as F
import time

torch.manual_seed(42)

def single_head_attention(Q, K, V):
    """
    Compute scaled dot-product attention.

    Args:
        Q: Query matrix  (seq_len, d_k)
        K: Key matrix    (seq_len, d_k)
        V: Value matrix  (seq_len, d_v)

    Returns:
        output: (seq_len, d_v) — the attended representations
        attn_weights: (seq_len, seq_len) — for visualization
    """
    d_k = Q.shape[-1]

    # Step 1: Compute raw attention scores — shape (seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)

    # Step 2: Apply softmax to get attention probabilities
    attn_weights = F.softmax(scores, dim=-1)

    # Step 3: Weighted sum of values
    output = torch.matmul(attn_weights, V)

    return output, attn_weights

# --- Quick sanity check ---
seq_len, d_k, d_v = 8, 16, 16
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_v)

output, attn_weights = single_head_attention(Q, K, V)

print("✅ Attention sanity check:")
print(f"   Q shape: {Q.shape}")
print(f"   K shape: {K.shape}")
print(f"   V shape: {V.shape}")
print(f"   Attention weights shape: {attn_weights.shape}")
print(f"   Output shape: {output.shape}")
print(f"   Attention weights sum to 1 (each row): {attn_weights.sum(dim=-1)}")

📊 Let us visualize what the attention weight matrix looks like. This is the n×n matrix that is the core of our scaling problem.

In [ ]:
# Visualize the attention weight matrix for a random sequence
torch.manual_seed(7)
seq_len_viz = 16
Q_viz = torch.randn(seq_len_viz, 32)
K_viz = torch.randn(seq_len_viz, 32)
V_viz = torch.randn(seq_len_viz, 32)

_, attn_viz = single_head_attention(Q_viz, K_viz, V_viz)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Attention heatmap ---
ax = axes[0]
im = ax.imshow(attn_viz.detach().numpy(), cmap='hot', aspect='auto')
ax.set_title(f'Attention Weight Matrix\n(seq_len={seq_len_viz}, shape = {seq_len_viz}×{seq_len_viz})',
             fontsize=13)
ax.set_xlabel('Key position (which token am I attending to?)', fontsize=11)
ax.set_ylabel('Query position (which token is asking?)', fontsize=11)
plt.colorbar(im, ax=ax, label='Attention weight')

# --- Annotate the growth of this matrix ---
ax2 = axes[1]
ns = np.array([4, 8, 16, 32, 64, 128, 256, 512, 1024])
matrix_sizes = ns ** 2   # number of elements in the n×n attention matrix
ax2.bar(range(len(ns)), matrix_sizes / 1e6,
        color=plt.cm.Reds(np.linspace(0.3, 0.9, len(ns))))
ax2.set_xticks(range(len(ns)))
ax2.set_xticklabels([str(n) for n in ns], rotation=45)
ax2.set_xlabel('Sequence Length n', fontsize=12)
ax2.set_ylabel('Attention Matrix Size (millions of elements)', fontsize=12)
ax2.set_title('The n×n Attention Matrix Grows Quadratically\n(and this gets computed FROM SCRATCH each step!)',
              fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

for i, (n, sz) in enumerate(zip(ns, matrix_sizes / 1e6)):
    if sz > 0.05:
        ax2.text(i, sz + 0.005 * matrix_sizes[-1]/1e6, f'{sz:.2f}M',
                 ha='center', va='bottom', fontsize=8, rotation=45)

plt.tight_layout()
plt.savefig('attention_matrix_visualization.png', dpi=120)
plt.show()
print("Notice: at seq_len=1024, the attention matrix alone has 1,048,576 elements.")
print("Every. Single. Generation. Step.")

### 4.2 The Naive Autoregressive Generator

Now let us build the naive inference loop — the one that recomputes everything from scratch at every step. We will include a **step counter** and a **FLOPs tracker** so we can see the waste accumulating in real time.

In [ ]:
class NaiveTransformerLayer(torch.nn.Module):
    """
    A single transformer layer with:
    - Linear projections to Q, K, V
    - Scaled dot-product attention
    - A simple output projection

    This is intentionally minimal — we want to isolate
    the attention cost, not hide it in library abstractions.
    """
    def __init__(self, d_model, d_k):
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k

        # These are the three projection matrices every transformer has
        self.W_Q = torch.nn.Linear(d_model, d_k, bias=False)
        self.W_K = torch.nn.Linear(d_model, d_k, bias=False)
        self.W_V = torch.nn.Linear(d_model, d_k, bias=False)
        self.W_out = torch.nn.Linear(d_k, d_model, bias=False)

        # We will track cost metadata
        self.step_flops = []

    def forward(self, x, track_cost=True):
        """
        x: (seq_len, d_model) — the full token sequence so far

        At each generation step, we receive the ENTIRE sequence
        and recompute EVERYTHING from scratch. That is the naive approach.
        """
        seq_len = x.shape[0]

        # --- Project to Q, K, V ---
        # Each projection: (seq_len, d_model) @ (d_model, d_k) → (seq_len, d_k)
        Q = self.W_Q(x)  # cost: seq_len * d_model * d_k
        K = self.W_K(x)  # cost: seq_len * d_model * d_k
        V = self.W_V(x)  # cost: seq_len * d_model * d_k

        # --- Attention ---
        # QK^T: (seq_len, d_k) @ (d_k, seq_len) → (seq_len, seq_len)
        # cost: seq_len^2 * d_k  ← this is the culprit
        attn_out, attn_weights = single_head_attention(Q, K, V)

        # --- Output projection ---
        output = self.W_out(attn_out)  # cost: seq_len * d_k * d_model

        if track_cost:
            # Approximate FLOPs: 2× multiplications for each multiply-add
            proj_flops = 3 * seq_len * self.d_model * self.d_k * 2
            attn_flops = seq_len**2 * self.d_k * 2   # the n^2 term!
            out_flops  = seq_len * self.d_k * self.d_model * 2
            total = proj_flops + attn_flops + out_flops
            self.step_flops.append({
                'seq_len': seq_len,
                'proj_flops': proj_flops,
                'attn_flops': attn_flops,
                'out_flops': out_flops,
                'total_flops': total
            })

        return output

# --- Test it ---
torch.manual_seed(42)
d_model, d_k = 64, 32
layer = NaiveTransformerLayer(d_model, d_k)

# Simulate a sequence of 10 tokens
x_test = torch.randn(10, d_model)
out = layer(x_test)
print(f"✅ Layer test passed. Input: {x_test.shape} → Output: {out.shape}")
print(f"   FLOPs breakdown at seq_len=10:")
info = layer.step_flops[-1]
print(f"   - Projection FLOPs: {info['proj_flops']:,}")
print(f"   - Attention FLOPs:  {info['attn_flops']:,}  ← this scales as n²")
print(f"   - Output FLOPs:     {info['out_flops']:,}")
print(f"   - Total:            {info['total_flops']:,}")

Now let us simulate the full naive generation loop — where we extend the sequence one token at a time and pay the full attention cost each time:

In [ ]:
def naive_generation_loop(layer, context_len, generate_len, d_model, device='cpu'):
    """
    Simulate naive autoregressive generation.

    At each step t, we:
      1. Feed the ENTIRE sequence (context + tokens generated so far) through the layer
      2. Take the last token's output as the new token embedding
      3. Append it to the sequence and repeat

    This is the O(n^3) disaster in code form.

    Returns:
        times_per_step: wall-clock time for each generation step (ms)
        flops_per_step: computed FLOPs for each generation step
        cumulative_flops: running total of FLOPs
    """
    layer.step_flops = []  # reset tracker

    # Start with a random context (simulating a prompt)
    torch.manual_seed(42)
    sequence = torch.randn(context_len, d_model)

    times_per_step = []
    flops_per_step = []

    print(f"Starting naive generation: context={context_len} tokens, generating {generate_len} more...")

    for step in range(generate_len):
        start = time.perf_counter()

        # THE NAIVE STEP: feed the ENTIRE sequence, get output for ALL positions
        with torch.no_grad():
            output = layer(sequence, track_cost=True)

        # Take only the LAST token's output as the newly generated token
        new_token = output[-1:, :]  # shape: (1, d_model)

        # Append to sequence
        sequence = torch.cat([sequence, new_token], dim=0)

        elapsed_ms = (time.perf_counter() - start) * 1000
        times_per_step.append(elapsed_ms)
        flops_per_step.append(layer.step_flops[-1]['total_flops'])

        if (step + 1) % 10 == 0:
            current_len = context_len + step + 1
            print(f"  Step {step+1:3d}: seq_len={current_len:4d}, "
                  f"time={elapsed_ms:.2f}ms, "
                  f"FLOPs={flops_per_step[-1]:,}")

    cumulative_flops = np.cumsum(flops_per_step)
    return times_per_step, flops_per_step, cumulative_flops


# Run the naive loop
d_model, d_k = 128, 64
layer = NaiveTransformerLayer(d_model, d_k)

times, flops_per_step, cum_flops = naive_generation_loop(
    layer,
    context_len=50,
    generate_len=50,
    d_model=d_model
)
print(f"\n✅ Generation complete.")
print(f"   First step time:   {times[0]:.3f} ms")
print(f"   Last step time:    {times[-1]:.3f} ms")
print(f"   Slowdown factor:   {times[-1]/times[0]:.1f}×")
print(f"   Total FLOPs used:  {sum(flops_per_step)/1e6:.2f} MFLOPs")

📊 Let us visualize the cost growth as generation proceeds:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

steps = np.arange(1, len(times) + 1)
seq_lens = 50 + steps  # context_len + step number

# --- Panel 1: Wall-clock time per step ---
ax = axes[0]
ax.plot(seq_lens, times, 'o-', color='crimson', lw=2, ms=4)
ax.set_xlabel('Sequence Length at This Step', fontsize=12)
ax.set_ylabel('Time per Generation Step (ms)', fontsize=12)
ax.set_title('Wall-Clock Time Grows\nWith Sequence Length', fontsize=12)
ax.grid(True, alpha=0.3)

# Fit a quadratic to illustrate growth
p = np.polyfit(seq_lens, times, 2)
fit_line = np.polyval(p, seq_lens)
ax.plot(seq_lens, fit_line, '--', color='orange', lw=2, label='Quadratic fit')
ax.legend(fontsize=10)

# --- Panel 2: FLOPs per step ---
ax = axes[1]
ax.plot(seq_lens, [f/1e3 for f in flops_per_step], 'o-', color='steelblue', lw=2, ms=4)
ax.set_xlabel('Sequence Length at This Step', fontsize=12)
ax.set_ylabel('FLOPs per Step (thousands)', fontsize=12)
ax.set_title('FLOPs Per Step Grows as n²\n(Quadratic per step)', fontsize=12)
ax.grid(True, alpha=0.3)

# --- Panel 3: Cumulative FLOPs ---
ax = axes[2]
ax.fill_between(seq_lens, [f/1e6 for f in cum_flops],
                alpha=0.3, color='purple')
ax.plot(seq_lens, [f/1e6 for f in cum_flops], '-', color='purple', lw=2.5)
ax.set_xlabel('Sequence Length at This Step', fontsize=12)
ax.set_ylabel('Cumulative FLOPs (millions)', fontsize=12)
ax.set_title('Total FLOPs Accumulate as n³\n(Cubic overall)', fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate the final cumulative cost
ax.annotate(f'Total: {cum_flops[-1]/1e6:.1f}M FLOPs',
            xy=(seq_lens[-1], cum_flops[-1]/1e6),
            xytext=(seq_lens[-5], cum_flops[-1]*0.75/1e6),
            arrowprops=dict(arrowstyle='->', color='purple'),
            fontsize=11, color='purple')

plt.suptitle('The Naive Generation Loop: Watching the Cost Grow in Real Time',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('naive_loop_cost.png', dpi=120)
plt.show()

## 5. 🔧 Your Turn

Now it is your turn to do something important. We have a function that measures the *total* naive inference cost across many different sequence lengths. But the FLOPs tracker inside `NaiveTransformerLayer` separately records the attention cost and the projection cost at every step.

Your task: implement a function that computes, for a given sequence length n, **what fraction of total FLOPs is spent on attention** (the n² term) versus projections (the n term). This will show us exactly when attention becomes the dominant cost.

In [ ]:
def compute_attention_fraction(layer, seq_lengths, d_model):
    """
    For each sequence length in seq_lengths, run one forward pass and
    compute what fraction of total FLOPs comes from attention (vs. projections).

    Args:
        layer: a NaiveTransformerLayer instance
        seq_lengths: list of integers (sequence lengths to test)
        d_model: embedding dimension

    Returns:
        attention_fractions: list of floats, one per seq_length
                             each value in [0, 1], representing fraction
                             of FLOPs spent in attention

    Hints:
        Step 1: For each n in seq_lengths, create a random tensor of shape (n, d_model)
        Step 2: Run a forward pass through the layer (with track_cost=True)
        Step 3: Retrieve the last entry in layer.step_flops
        Step 4: Compute attn_flops / total_flops for that entry
        Step 5: Append to your result list and reset layer.step_flops = []
    """
    # ============ TODO ============
    # Step 1: Initialize an empty list to store results
    attention_fractions = []

    # Step 2: Loop over each sequence length
    for n in seq_lengths:
        # Step 3: Create a random input tensor of shape (n, d_model)
        x = ???

        # Step 4: Run forward pass (use torch.no_grad() for efficiency)
        with torch.no_grad():
            ???

        # Step 5: Get the last recorded FLOPs info
        info = ???

        # Step 6: Compute the fraction and append it
        fraction = ???
        attention_fractions.append(fraction)

        # Step 7: Reset the tracker for the next iteration
        layer.step_flops = []

    # ============ END TODO ========
    return attention_fractions


# ✅ Verification cell — run this after implementing the function above
# (Do not peek at the expected output until you have written your solution!)

try:
    torch.manual_seed(0)
    test_layer = NaiveTransformerLayer(d_model=64, d_k=32)
    test_lens = [10, 50, 100]
    fracs = compute_attention_fraction(test_layer, test_lens, d_model=64)

    assert len(fracs) == 3, "Should return one value per sequence length"
    assert all(0 <= f <= 1 for f in fracs), "Fractions must be in [0, 1]"
    assert fracs[2] > fracs[0], "Attention fraction should INCREASE with sequence length"

    print("✅ All assertions passed!")
    print(f"   Attention fraction at n=10:  {fracs[0]:.3f} ({fracs[0]*100:.1f}% of FLOPs)")
    print(f"   Attention fraction at n=50:  {fracs[1]:.3f} ({fracs[1]*100:.1f}% of FLOPs)")
    print(f"   Attention fraction at n=100: {fracs[2]:.3f} ({fracs[2]*100:.1f}% of FLOPs)")
    print()
    print("💡 Notice: as n grows, attention dominates more and more of the total cost.")

except NameError:
    print("⚠️  Implement compute_attention_fraction() above first, then re-run this cell.")
except AssertionError as e:
    print(f"❌ Assertion failed: {e}")
    print("   Double-check your fraction calculation.")

## 6. Putting It All Together

Let us now run the complete experiment: measure naive inference time across a wide range of sequence lengths, confirm the O(n³) scaling empirically, and visualize the breakdown between attention cost and projection cost.

In [ ]:
# Full experiment: naive inference scaling across many sequence lengths
torch.manual_seed(42)

d_model, d_k = 128, 64
layer_full = NaiveTransformerLayer(d_model, d_k)

test_seq_lengths = [16, 32, 64, 128, 256, 512, 768, 1024]
n_trials = 5  # average over multiple trials for stability

print("Running timing experiment...")
print(f"{'Seq Len':>10} {'Time (ms)':>12} {'Attn FLOPs':>15} {'Total FLOPs':>15} {'Attn %':>10}")
print("-" * 65)

results = []
for n in test_seq_lengths:
    x = torch.randn(n, d_model)
    layer_full.step_flops = []

    # Warm-up run
    with torch.no_grad():
        _ = layer_full(x, track_cost=False)

    # Timed runs
    trial_times = []
    for _ in range(n_trials):
        layer_full.step_flops = []
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = layer_full(x, track_cost=True)
        trial_times.append((time.perf_counter() - t0) * 1000)

    avg_time = np.mean(trial_times)
    info = layer_full.step_flops[-1]
    attn_pct = 100 * info['attn_flops'] / info['total_flops']

    results.append({
        'n': n,
        'time_ms': avg_time,
        'attn_flops': info['attn_flops'],
        'proj_flops': info['proj_flops'],
        'total_flops': info['total_flops'],
        'attn_pct': attn_pct
    })

    print(f"{n:>10} {avg_time:>12.3f} {info['attn_flops']:>15,} "
          f"{info['total_flops']:>15,} {attn_pct:>9.1f}%")

print(f"\n✅ Experiment complete.")
print(f"   Slowdown from n={test_seq_lengths[0]} to n={test_seq_lengths[-1]}: "
      f"{results[-1]['time_ms']/results[0]['time_ms']:.1f}×")
expected_cubic = (test_seq_lengths[-1] / test_seq_lengths[0])**3
print(f"   Expected under O(n³): {expected_cubic:.0f}×")

📊 The complete picture — let us make this beautiful and clear:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ns = [r['n'] for r in results]
times_ms = [r['time_ms'] for r in results]
attn_flops = [r['attn_flops'] for r in results]
proj_flops = [r['proj_flops'] for r in results]
total_flops = [r['total_flops'] for r in results]
attn_pcts = [r['attn_pct'] for r in results]

# --- Panel 1: Timing on log-log scale with slope reference ---
ax = axes[0, 0]
ax.loglog(ns, times_ms, 'o-', color='crimson', lw=2.5, ms=9, label='Measured time')
# Fit a power law
log_n = np.log(ns)
log_t = np.log(times_ms)
slope, intercept = np.polyfit(log_n, log_t, 1)
fitted = np.exp(intercept) * np.array(ns)**slope
ax.loglog(ns, fitted, '--', color='orange', lw=2,
          label=f'Power law fit: n^{slope:.2f}')
ax.set_xlabel('Sequence Length n', fontsize=12)
ax.set_ylabel('Time per Forward Pass (ms)', fontsize=12)
ax.set_title(f'Empirical Scaling: Fitted Exponent = {slope:.2f}', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.text(0.05, 0.85, f'Slope ≈ {slope:.2f}\n(theory predicts ≈ 2)',
        transform=ax.transAxes, fontsize=11, color='orange',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Panel 2: Stacked bar: Attention vs Projection FLOPs ---
ax = axes[0, 1]
width = 0.6
p1 = ax.bar(range(len(ns)), [f/1e6 for f in attn_flops],
            width, label='Attention FLOPs (n² term)', color='crimson', alpha=0.85)
p2 = ax.bar(range(len(ns)), [f/1e6 for f in proj_flops],
            width, bottom=[f/1e6 for f in attn_flops],
            label='Projection FLOPs (n term)', color='steelblue', alpha=0.85)
ax.set_xticks(range(len(ns)))
ax.set_xticklabels([str(n) for n in ns], rotation=45)
ax.set_xlabel('Sequence Length n', fontsize=12)
ax.set_ylabel('FLOPs (millions)', fontsize=12)
ax.set_title('FLOPs Breakdown: Attention vs. Projections', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# --- Panel 3: Attention fraction ---
ax = axes[1, 0]
ax.plot(ns, attn_pcts, 's-', color='purple', lw=2.5, ms=9)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
ax.fill_between(ns, attn_pcts, alpha=0.2, color='purple')
ax.set_xlabel('Sequence Length n', fontsize=12)
ax.set_ylabel('Attention FLOPs as % of Total', fontsize=12)
ax.set_title('Attention Dominates More as n Grows', fontsize=12)
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
for i, (n, pct) in enumerate(zip(ns, attn_pcts)):
    ax.annotate(f'{pct:.0f}%', (n, pct + 2), ha='center', fontsize=9)

# --- Panel 4: Cumulative cost of full generation ---
ax = axes[1, 1]
gen_lengths = [16, 32, 64, 128, 256, 512]
colors_gen = plt.cm.Reds(np.linspace(0.3, 0.9, len(gen_lengths)))

for gen_len, col in zip(gen_lengths, colors_gen):
    # Simulate generating gen_len tokens from scratch
    # Total FLOPs = sum_{t=1}^{gen_len} t^2 * d_k * 2 (attention only, dominant term)
    cumulative = np.cumsum([(t**2 * d_k * 2) / 1e6 for t in range(1, gen_len+1)])
    ax.plot(range(1, gen_len+1), cumulative, '-', color=col, lw=2,
            label=f'Generate {gen_len} tokens')

ax.set_xlabel('Number of Tokens Generated', fontsize=12)
ax.set_ylabel('Cumulative Attention FLOPs (MFLOPs)', fontsize=12)
ax.set_title('Cumulative Cost of Naive Generation\n(Attention FLOPs only)', fontsize=12)
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)

plt.suptitle('The O(n³) Disaster: Complete Empirical Picture',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('complete_naive_analysis.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. The Waste We Can Quantify

The naive loop is not just slow in abstract O(n) terms — we can put a precise number on exactly how much redundant work it does. Let us compute the **redundancy ratio**: the ratio of total FLOPs spent in naive generation to the theoretical minimum (if we could somehow avoid all recomputation).

In [ ]:
def compute_redundancy_ratio(n_generate, n_context, d_k):
    """
    Compute how much extra work the naive loop does compared
    to the theoretical minimum.

    Theoretical minimum: compute attention ONCE for the final full sequence.
    Naive cost: compute attention fresh at every step.

    Returns:
        redundancy_ratio: naive_total / optimal_cost
        naive_total_flops: total FLOPs in naive approach
        optimal_flops: FLOPs if we could do it optimally (one pass)
    """
    total_len = n_context + n_generate

    # Naive: at step t (sequence length = n_context + t),
    # attention cost is (n_context + t)^2 * d_k * 2
    naive_total = sum(
        (n_context + t)**2 * d_k * 2
        for t in range(1, n_generate + 1)
    )

    # Theoretical minimum: one full attention pass over the entire sequence
    optimal = total_len**2 * d_k * 2

    redundancy = naive_total / optimal

    return redundancy, naive_total, optimal


# Compute for different generation scenarios
print("=" * 70)
print(f"{'Scenario':>30} {'Naive MFLOPs':>15} {'Optimal MFLOPs':>15} {'Waste Factor':>12}")
print("=" * 70)

scenarios = [
    (50, 50, "Short doc, short gen"),
    (500, 100, "Medium doc, short gen"),
    (1000, 500, "Long doc, medium gen"),
    (2000, 1000, "Legal contract example"),
    (4000, 2000, "Very long document"),
]

for n_ctx, n_gen, label in scenarios:
    ratio, naive, opt = compute_redundancy_ratio(n_gen, n_ctx, d_k=64)
    print(f"{label:>30} {naive/1e6:>15.1f} {opt/1e6:>15.1f} {ratio:>11.1f}×")

print("=" * 70)
print()
print("💡 The 'legal contract example' from the article (2000 tokens, 1000 generated):")
ratio, _, _ = compute_redundancy_ratio(1000, 2000, d_k=64)
print(f"   The naive loop does {ratio:.1f}× MORE work than the theoretical minimum.")
print(f"   This is the waste that the KV cache eliminates.")

📊 Let us plot the redundancy ratio as a function of generation length — this is the clearest possible illustration of the problem:

In [ ]:
n_ctx = 500  # fixed context length
n_generates = np.arange(10, 1001, 10)
ratios = [compute_redundancy_ratio(int(g), n_ctx, d_k=64)[0] for g in n_generates]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel 1: Redundancy ratio vs generation length ---
ax = axes[0]
ax.plot(n_generates, ratios, '-', color='crimson', lw=2.5)
ax.fill_between(n_generates, 1, ratios, alpha=0.2, color='crimson',
                label='Wasted computation')
ax.axhline(y=1, color='green', linestyle='--', lw=2, label='Theoretical minimum (1×)')
ax.set_xlabel('Number of Tokens to Generate', fontsize=12)
ax.set_ylabel('Redundancy Ratio (naive / optimal)', fontsize=12)
ax.set_title(f'Naive Inference Redundancy\n(Context = {n_ctx} tokens fixed)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.annotate(f'{ratios[-1]:.1f}× redundant\nat 1000 generated tokens',
            xy=(1000, ratios[-1]),
            xytext=(700, ratios[-1]*0.85),
            arrowprops=dict(arrowstyle='->', color='crimson'),
            fontsize=11, color='crimson')

# --- Panel 2: Where the time goes (pie chart for a concrete scenario) ---
ax = axes[1]
n_ctx_pie, n_gen_pie = 2000, 1000
total_len = n_ctx_pie + n_gen_pie

# FLOPs per step
step_flops_breakdown = [(n_ctx_pie + t)**2 * 64 * 2 for t in range(1, n_gen_pie+1)]
step_nums = np.arange(1, n_gen_pie+1)

# Show early vs late generation cost
early_cost = sum(step_flops_breakdown[:100])
mid_cost   = sum(step_flops_breakdown[100:500])
late_cost  = sum(step_flops_breakdown[500:])
total = early_cost + mid_cost + late_cost

sizes = [early_cost/total*100, mid_cost/total*100, late_cost/total*100]
labels = [
    f'Early gen (steps 1-100)\n{early_cost/total*100:.1f}%',
    f'Mid gen (steps 101-500)\n{mid_cost/total*100:.1f}%',
    f'Late gen (steps 501-1000)\n{late_cost/total*100:.1f}%'
]
colors = ['#fee8d0', '#f5956b', '#c0392b']
explode = (0, 0, 0.05)

wedges, texts = ax.pie(sizes, labels=labels, colors=colors, explode=explode,
                        startangle=90, textprops={'fontsize': 10})
ax.set_title(f'Where FLOPs Are Spent\n(ctx={n_ctx_pie}, generate={n_gen_pie} tokens)',
             fontsize=12)
ax.text(0, -1.4, 'Late generation steps dominate because seq_len is largest then.',
        ha='center', fontsize=10, style='italic')

plt.suptitle('The Compounding Cost of Naive Autoregressive Inference',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('redundancy_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. 🎯 Final Output — The O(n³) Case Made Undeniable

Let us produce the definitive visualization: a single, publication-quality figure that tells the complete story of naive inference scaling. This is the screenshot you will want to keep.

In [ ]:
fig = plt.figure(figsize=(16, 10))

# Use a grid layout
gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :2])  # Wide top-left: the main scaling plot
ax2 = fig.add_subplot(gs[0, 2])   # Top-right: FLOPs breakdown
ax3 = fig.add_subplot(gs[1, 0])   # Bottom-left: per-step cost
ax4 = fig.add_subplot(gs[1, 1])   # Bottom-middle: cumulative comparison
ax5 = fig.add_subplot(gs[1, 2])   # Bottom-right: the human cost

# ---- Panel 1: Main O(n^3) Scaling Demonstration ----
n_range = np.logspace(1.5, 3.5, 200)
d_k_demo = 64

linear_ref    = (n_range / n_range[0])
quadratic_ref = (n_range / n_range[0])**2
cubic_ref     = (n_range / n_range[0])**3

ax1.loglog(n_range, linear_ref, '--', color='green', lw=1.5, alpha=0.6, label='O(n) — linear')
ax1.loglog(n_range, quadratic_ref, '--', color='steelblue', lw=1.5, alpha=0.6, label='O(n²) — per-step cost')
ax1.loglog(n_range, cubic_ref, '-', color='crimson', lw=3, label='O(n³) — naive total cost')

# Shade the cubic region
ax1.fill_between(n_range, quadratic_ref, cubic_ref, alpha=0.12, color='crimson')

# Annotate key sequence lengths
for n_mark, label_str in [(100, '100 tokens\n(short reply)'),
                           (1000, '1K tokens\n(1 page)'),
                           (10000, '10K tokens\n(legal doc)')]:
    if n_mark <= n_range[-1]:
        y_val = (n_mark / n_range[0])**3
        ax1.scatter([n_mark], [y_val], color='crimson', s=80, zorder=5)
        ax1.annotate(label_str, (n_mark, y_val), textcoords='offset points',
                     xytext=(10, 5), fontsize=9, color='crimson')

ax1.set_xlabel('Sequence Length n (tokens)', fontsize=13)
ax1.set_ylabel('Relative Computational Cost', fontsize=13)
ax1.set_title('Naive Inference: O(n³) Total Cost — The Core Problem',
              fontsize=14, fontweight='bold')
ax1.legend(fontsize=11, loc='upper left')
ax1.grid(True, alpha=0.25, which='both')

# Add a text box explaining the jump
ax1.text(0.55, 0.15,
         'Doubling sequence length → 8× the compute\n'
         'Tripling sequence length → 27× the compute\n'
         'This makes long-context inference catastrophically slow.',
         transform=ax1.transAxes, fontsize=10,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff3f3',
                   edgecolor='crimson', alpha=0.9))

# ---- Panel 2: Attention vs projection breakdown ----
ns_bar = [64, 128, 256, 512, 1024]
attn_f = [n**2 * d_k_demo * 2 / 1e6 for n in ns_bar]
proj_f = [n * 128 * d_k_demo * 2 / 1e6 for n in ns_bar]  # d_model=128

bars1 = ax2.bar(range(len(ns_bar)), attn_f, label='Attention (n²)', color='crimson', alpha=0.85)
bars2 = ax2.bar(range(len(ns_bar)), proj_f, bottom=attn_f,
                label='Projections (n)', color='steelblue', alpha=0.85)
ax2.set_xticks(range(len(ns_bar)))
ax2.set_xticklabels([str(n) for n in ns_bar])
ax2.set_xlabel('Sequence Length', fontsize=11)
ax2.set_ylabel('MFLOPs per Step', fontsize=11)
ax2.set_title('Attention Dominates\nProjections', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

# ---- Panel 3: Per-step time growth ----
step_lengths = np.arange(10, 201, 5)
per_step_cost = step_lengths**2 * d_k_demo * 2 / 1e3  # in KFLOPs

ax3.plot(step_lengths, per_step_cost, '-', color='orange', lw=2.5)
ax3.fill_between(step_lengths, per_step_cost, alpha=0.2, color='orange')
ax3.set_xlabel('Sequence Length at Step t', fontsize=11)
ax3.set_ylabel('Attention Cost (KFLOPs)', fontsize=11)
ax3.set_title('Cost Per Step Grows as n²\n(Quadratic per step)', fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3)

# ---- Panel 4: Cumulative cost comparison ----
gen_len = 200
steps_arr = np.arange(1, gen_len+1)
context = 50

naive_cumulative = np.cumsum([(context + t)**2 * d_k_demo * 2 for t in steps_arr]) / 1e6
optimal_constant = np.full_like(naive_cumulative,
                                (context + gen_len)**2 * d_k_demo * 2 / 1e6)

ax4.plot(steps_arr, naive_cumulative, '-', color='crimson', lw=2.5, label='Naive (cumulative)')
ax4.plot(steps_arr, optimal_constant, '--', color='green', lw=2, label='One-pass (optimal)')
ax4.fill_between(steps_arr, optimal_constant, naive_cumulative,
                 alpha=0.2, color='crimson', label='Wasted FLOPs')
ax4.set_xlabel('Tokens Generated', fontsize=11)
ax4.set_ylabel('Cumulative Attention FLOPs (MFLOPs)', fontsize=11)
ax4.set_title(f'Naive vs. Optimal\n(ctx={context}, d_k={d_k_demo})', fontsize=11, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

# ---- Panel 5: "What does this mean in human terms?" ----
ax5.axis('off')
context_lengths_human = [500, 1000, 2000, 4000]
generate_lengths_human = [200, 500, 1000, 2000]

table_data = []
for ctx, gen in zip(context_lengths_human, generate_lengths_human):
    ratio, _, _ = compute_redundancy_ratio(gen, ctx, d_k=64)
    table_data.append([f'{ctx}+{gen}', f'{ratio:.1f}×'])

table = ax5.table(
    cellText=table_data,
    colLabels=['Ctx + Gen Tokens', 'Wasted Work'],
    cellLoc='center',
    loc='center',
    bbox=[0.05, 0.25, 0.9, 0.55]
)
table.auto_set_font_size(False)
table.set_fontsize(11)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#c0392b')
        cell.set_text_props(color='white', fontweight='bold')
    elif col == 1:
        try:
            val = float(cell.get_text().get_text().replace('×', ''))
            intensity = min(val / 5.0, 1.0)
            cell.set_facecolor(plt.cm.Reds(0.2 + 0.5 * intensity))
        except:
            pass

ax5.set_title('Redundancy in Real Scenarios', fontsize=12, fontweight='bold', pad=20)
ax5.text(0.5, 0.1, '← KV caching eliminates this waste entirely',
         ha='center', fontsize=10, transform=ax5.transAxes,
         color='green', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#eafaf1', edgecolor='green'))

# ---- Final title ----
fig.suptitle(
    'Section 2: The Naive Inference Loop — An O(n³) Disaster\n'
    'Vizuara | KV Cache Series',
    fontsize=16, fontweight='bold', y=1.01
)

plt.savefig('section2_final_output.png', dpi=140, bbox_inches='tight',
            facecolor='white')
plt.show()
print("🎯 Final output saved as 'section2_final_output.png'")
print("   This tells the complete story of naive inference scaling.")

## 9. Reflection and Next Steps

Let us take stock of everything we have established in this section.

### What We Proved

We started with a simple question — *why is naive inference so expensive?* — and we answered it completely, from first principles:

1. **Attention is an n×n operation.** For every generation step, the transformer must compute a full n×n matrix of pairwise token relationships. This costs O(n²·d) FLOPs per step.

2. **Autoregressive generation compounds this.** To generate n tokens, we pay the attention cost n times — once per token. The sum ∑ t² from t=1 to n grows as n³/3, giving us the O(n³) total.

3. **The waste is measurable.** For a realistic scenario (2,000-token context, 1,000-token generation), the naive loop does more than 3× the computation that would theoretically be necessary — and that ratio only grows with sequence length.

4. **Attention dominates projections.** For long sequences, the n² attention term dwarfs the n projection terms. This tells us exactly where an optimization must focus.

### 🤔 Reflection Questions

Think carefully about these before moving on:

> **Q1:** We showed that naive generation costs O(n³). But in a single forward pass through a transformer (not generation — just *encoding* a sequence), the cost is O(n²·d). When does it make sense to generate token-by-token, and when does it make sense to process the whole sequence at once?

> **Q2:** Our analysis assumed a single attention head. Real transformers have many heads (e.g., 32 in GPT-3). How does the number of heads affect the total cost? Does it change the *scaling* or just the *constant*?

> **Q3:** The n×n attention matrix is computed but then discarded at every step in the naive loop. What information would you need to *save* from one step to the next to avoid recomputing it? (This is the key intuition for understanding the KV cache.)

> **Q4:** We measured that the naive approach becomes dramatically slower for long sequences. But in practice, many deployed systems handle long contexts reasonably well. What might explain this — what tricks might they be using?

### 🏆 Optional Challenges

If you want to go deeper before the next section, here are three challenges:

**Challenge 1 — Multi-head attention cost:**
Extend `NaiveTransformerLayer` to support `n_heads` attention heads (each with dimension `d_k // n_heads`). Verify that the total FLOPs are the same as single-head attention with the same total dimension. (Hint: they should be — multi-head attention costs the same FLOPs as single-head when you account for the split dimensions.)

**Challenge 2 — The causal mask:**
In decoder-only models (like GPT), attention is *causal*: token t can only attend to tokens 1...t, not future tokens. This means the effective attention matrix is lower-triangular. Does this change the asymptotic scaling? Implement causal masking in `single_head_attention` and re-run the FLOPs analysis.

**Challenge 3 — Profiling with PyTorch:**
Use `torch.profiler` to profile the `NaiveTransformerLayer` forward pass at seq_len=512. What fraction of wall-clock time is the `torch.matmul` for QK^T versus the linear projections? Does this match our FLOPs prediction?

In [ ]:
# Starter code for Challenge 2: Causal Masking
def causal_attention(Q, K, V):
    """
    Implement causal (masked) attention.
    Token at position i can only attend to positions j <= i.

    Args:
        Q, K, V: (seq_len, d_k) tensors

    Returns:
        output: (seq_len, d_k)
        attn_weights: (seq_len, seq_len) — should be lower triangular
    """
    seq_len, d_k = Q.shape

    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)

    # Create causal mask: positions where j > i should be -inf
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    scores = scores.masked_fill(mask, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)
    # Note: positions with -inf become 0 after softmax
    # Replace NaN (from 0/0 at the first position in some implementations)
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Quick test
torch.manual_seed(0)
seq_len = 6
Q_c = torch.randn(seq_len, 16)
K_c = torch.randn(seq_len, 16)
V_c = torch.randn(seq_len, 16)

out_c, weights_c = causal_attention(Q_c, K_c, V_c)

print("Causal attention weight matrix (should be lower triangular):")
print(weights_c.detach().numpy().round(3))
print()
print("✅ Upper triangle is all zeros (causal masking works).")
print()
print("💡 Notice: causal masking does NOT change the asymptotic FLOPs.")
print("   We still compute the full n×n matrix first, then mask.")
print("   FlashAttention exploits this structure, but that is a later story...")
print()
print("=" * 60)
print("🎯 Next Section: The KV Cache — The Brilliant Fix")
print("   We know exactly what the problem is.")
print("   We know exactly which computations are being repeated.")
print("   In Section 3, we will see how to save and reuse them.")
print("=" * 60)